Mosaic AI Agent Framework

In [0]:
%pip install -U -qqqq mlflow databricks-openai databricks-agents
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.12.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# The snippet below tries to pick the first LLM API available in your Databricks workspace
# from a set of candidates. You can override and simplify it
# to just specify LLM_ENDPOINT_NAME.
LLM_ENDPOINT_NAME = None

from databricks_openai import DatabricksOpenAI
def is_endpoint_available(endpoint_name):
  try:
    client = DatabricksOpenAI()
    client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
    return True
  except Exception:
    return False
  
for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
    if is_endpoint_available(candidate_endpoint_name):
      LLM_ENDPOINT_NAME = candidate_endpoint_name
assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME"

In [0]:
from databricks.sdk import WorkspaceClient

# Refresh Unity Catalog metadata for system.ai functions
# This ensures the python_exec tool is discoverable in your workspace
w = WorkspaceClient()
list(w.schemas.list("system"))
w.schemas.get("system.ai")
_ = list(w.functions.list(catalog_name="system", schema_name="ai"))

In [0]:
import json
import mlflow
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient, DatabricksOpenAI
# Import MLflow utilities for converting from chat completions to responses API format
from mlflow.types.responses import output_to_responses_items_stream, create_function_call_output_item

# Automatically log traces from LLM calls for ease of debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to talk to Databricks model serving endpoints
# We'll use this to query an LLM in our agent
openai_client = DatabricksOpenAI()

# Load Databricks built-in tools (a stateless Python code interpreter tool)
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(
    function_names=["system.ai.python_exec"], client=client
).tools
for tool in builtin_tools:
    del tool["function"]["strict"]


def call_tool(tool_name, parameters):
    if tool_name == "system__ai__python_exec":
        return DatabricksFunctionClient().execute_function(
            "system.ai.python_exec", parameters=parameters
        ).value
    raise ValueError(f"Unknown tool: {tool_name}")

def call_llm(prompt):
    for chunk in openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=builtin_tools,
        stream=True
    ):
        yield chunk.to_dict()

def run_agent(prompt):
    """
    Send a user prompt to the LLM, and yield LLM + tool call responses
    The LLM is allowed to call the code interpreter tool if needed, to respond to the user
    """
    # Convert output into Responses API-compatible events
    for chunk in output_to_responses_items_stream(call_llm(prompt)):
        yield chunk.model_dump(exclude_none=True)
    # If the model executed a tool, call it and yield the tool call output in Responses API format
    if chunk.item.get('type') == 'function_call':
        tool_name = chunk.item["name"]
        tool_args = json.loads(chunk.item["arguments"])
        tool_result = call_tool(tool_name, tool_args)
        yield {"type": "response.output_item.done", "item": create_function_call_output_item(call_id=chunk.item["call_id"], output=tool_result)}

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


In [0]:
for output_chunk in run_agent("What is the square root of 429?"):
    print(output_chunk)


Provider List: https://docs.litellm.ai/docs/providers

{'type': 'response.output_item.done', 'item': {'type': 'function_call', 'id': 'chatcmpl_7f5ca9ef-3c2c-4155-a4b4-e3ea7612072a', 'call_id': 'call_1de92d08-c822-4d4b-ac6d-b283e7337779', 'name': 'system__ai__python_exec', 'arguments': '{"code": "import math\\nprint(math.sqrt(429))"}'}}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


{'type': 'response.output_item.done', 'item': {'type': 'function_call_output', 'call_id': 'call_1de92d08-c822-4d4b-ac6d-b283e7337779', 'output': '20.71231517720798\n'}}


Trace(trace_id=tr-0ec843d6fc0832108827728e56677994)

In [0]:
import uuid
import mlflow
from typing import Any, Optional, Generator

from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse, ResponsesAgentStreamEvent, output_to_responses_items_stream

mlflow.openai.autolog()

class QuickstartAgent(ResponsesAgent):
    def predict_stream(self, request: ResponsesAgentRequest): 
        # Extract the user's prompt from the request
        prompt = request.input[-1].content
        # Stream response items from our agent
        for chunk in run_agent(prompt):
            yield ResponsesAgentStreamEvent(**chunk)

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs)


In [0]:
from mlflow.types.responses import ResponsesAgentRequest

AGENT = QuickstartAgent()

# Create a proper ResponsesAgentRequest with input field
request = ResponsesAgentRequest(
    input=[
            {
                "role": "user", 
                "content": "What's the square root of 429?"
            }
    ]
)

for event in AGENT.predict_stream(request):
    print(event)



Provider List: https://docs.litellm.ai/docs/providers

type='response.output_item.done' custom_outputs=None item={'type': 'function_call', 'id': 'chatcmpl_88f64b7a-a049-4185-b31e-56f586ea9125', 'call_id': 'call_484482fb-23ec-4f85-95ee-d0e165361812', 'name': 'system__ai__python_exec', 'arguments': '{ "code": "import math\\nprint(math.sqrt(429))" }'}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


type='response.output_item.done' custom_outputs=None item={'type': 'function_call_output', 'call_id': 'call_484482fb-23ec-4f85-95ee-d0e165361812', 'output': '20.71231517720798\n'}


Trace(trace_id=tr-d846814c212994015e338ec1aefe6d71)

In [0]:
%%writefile quickstart_agent.py

import json
import uuid
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient, DatabricksOpenAI
from typing import Any, Optional, Generator

import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentStreamEvent, ResponsesAgentResponse, output_to_responses_items_stream, create_function_call_output_item


# Get an OpenAI client configured to talk to Databricks model serving endpoints
# We'll use this to query an LLM in our agent
openai_client = DatabricksOpenAI()

# The snippet below tries to pick the first LLM API available in your Databricks workspace
# from a set of candidates. You can override and simplify it
# to just specify LLM_ENDPOINT_NAME.
LLM_ENDPOINT_NAME = None

def is_endpoint_available(endpoint_name):
  try:
    client = DatabricksOpenAI()
    client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
    return True
  except Exception:
    return False

for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
    if is_endpoint_available(candidate_endpoint_name):
      LLM_ENDPOINT_NAME = candidate_endpoint_name
assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME"

# Automatically log traces from LLM calls for ease of debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to talk to Databricks model serving endpoints
# We'll use this to query an LLM in our agent
openai_client = DatabricksOpenAI()

# Load Databricks built-in tools (a stateless Python code interpreter tool)
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(
    function_names=["system.ai.python_exec"], client=client
).tools
for tool in builtin_tools:
    del tool["function"]["strict"]


def call_tool(tool_name, parameters):
    if tool_name == "system__ai__python_exec":
        return DatabricksFunctionClient().execute_function(
            "system.ai.python_exec", parameters=parameters
        ).value
    raise ValueError(f"Unknown tool: {tool_name}")

def call_llm(prompt):
    for chunk in openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=builtin_tools,
        stream=True
    ):
        yield chunk.to_dict()


def run_agent(prompt):
    """
    Send a user prompt to the LLM, and yield LLM + tool call responses
    The LLM is allowed to call the code interpreter tool if needed, to respond to the user
    """
    # Convert output into Responses API-compatible events
    for chunk in output_to_responses_items_stream(call_llm(prompt)):
        yield chunk.model_dump(exclude_none=True)
    # If the model executed a tool, call it and yield the tool call output in Responses API format
    if chunk.item.get('type') == 'function_call':
        tool_name = chunk.item["name"]
        tool_args = json.loads(chunk.item["arguments"])
        tool_result = call_tool(tool_name, tool_args)
        yield {"type": "response.output_item.done", "item": create_function_call_output_item(call_id=chunk.item["call_id"], output=tool_result)}


class QuickstartAgent(ResponsesAgent):
    def predict_stream(self, request: ResponsesAgentRequest): 
        # Extract the user's prompt from the request
        prompt = request.input[-1].content
        # Stream response items from our agent
        for chunk in run_agent(prompt):
            yield ResponsesAgentStreamEvent(**chunk)

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs)

AGENT = QuickstartAgent()
mlflow.models.set_model(AGENT)

Writing quickstart_agent.py


In [0]:
dbutils.library.restartPython()

In [0]:
# Test the ResponsesAgent implementation
from quickstart_agent import QuickstartAgent, LLM_ENDPOINT_NAME
from mlflow.types.responses import ResponsesAgentRequest

print(f"Using LLM endpoint: {LLM_ENDPOINT_NAME}")

# Create agent instance
agent = QuickstartAgent()

# Create test request - input should be a list of messages
request = ResponsesAgentRequest(
    input=[
        {
            "role": "user", 
            "content": "What's the square root of 144?"
        }
    ]
)

# Test the agent
print("\nTesting agent...")
response = agent.predict(request)
print(f"\nAgent response:")
for i, output_item in enumerate(response.output):
    print(f"Item {i+1}: Type={output_item.type}")
    print(f" {output_item}")


/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Using LLM endpoint: databricks-meta-llama-3-3-70b-instruct

Testing agent...

Provider List: https://docs.litellm.ai/docs/providers



/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)



Agent response:
Item 1: Type=function_call
 type='function_call' id='chatcmpl_696741fb-daf3-4bcb-8ceb-04d192d66244' call_id='call_a73c1298-8eb7-42f6-a9ff-2db1a919888a' name='system__ai__python_exec' arguments='{ "code": "import math\\nprint(math.sqrt(144))" }'
Item 2: Type=function_call_output
 type='function_call_output' call_id='call_a73c1298-8eb7-42f6-a9ff-2db1a919888a' output='12.0\n'


Trace(trace_id=tr-53ad88d2fa29d61388f1b0b811cc7804)

In [0]:
import mlflow
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution
from quickstart_agent import LLM_ENDPOINT_NAME

# Register the model to the workspace default catalog.
# Specify a catalog (e.g. "main") and schema name (e.g. "custom_schema") if needed,
# in order to register the agent to a different location
default_catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
catalog_name = default_catalog_name if default_catalog_name != "hive_metastore" else "main"
schema_name = "default"
registered_model_name = f"{catalog_name}.{schema_name}.quickstart_agent"

# Specify Databricks product resources that the agent needs access to (our builtin python
# code interpreter tool and LLM serving endpoint), so that Databricks can automatically
# configure authentication for the agent to access these resources when it's deployed.
resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    DatabricksFunction(function_name="system.ai.python_exec"),
]

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="quickstart_agent.py",
        extra_pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}"
        ],
        resources=resources,
        registered_model_name=registered_model_name,
    )

🔗 View Logged Model at: https://dbc-561b22b1-578d.cloud.databricks.com/ml/experiments/312100876764827/models/m-a8ec5b547945453799c528b85cc24af0?o=7474649359993770
2026/02/28 21:53:01 WARNING mlflow.tracing.fluent: Failed to start span DatabricksCompletions: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/02/28 21:53:01 WARNING mlflow.tracing.fluent: Failed to start span DatabricksCompletions: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)
2026/02/28 21:53:08 INFO mlflow.pyfunc: 

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.quickstart_agent': https://dbc-561b22b1-578d.cloud.databricks.com/explore/data/models/workspace/default/quickstart_agent/version/1?o=7474649359993770


In [0]:
from databricks import agents

deployment_info = agents.deploy(
    model_name=registered_model_name,
    model_version=logged_agent_info.registered_model_version,
    scale_to_zero=True,
    deploy_feedback_model=False
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47da88f-4669-4acf-a01a-aded74aa2e6d/lib/python3.10/site-packages/databricks/agents/deployments.py:641: UserWarning: This endpoint is being deployed without a feedback model, which has been deprecated.
For more information, see: https://docs.databricks.com/aws/en/generative-ai/agent-framework/feedback-model
  warnings.warn(



    Deployment of workspace.default.quickstart_agent version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://dbc-561b22b1-578d.cloud.databricks.com/ml/endpoints/agents_workspace-default-quickstart_agent/?o=7474649359993770
    Review App: https://dbc-561b22b1-578d.cloud.databricks.com/ml/review-v2/9cc1d98dfb1a44c78a133ab57085c10c/chat?o=7474649359993770

You can refer back to the links above from the endpoint detail page at https://dbc-561b22b1-578d.cloud.databricks.com/ml/endpoints/agents_workspace-default-quickstart_agent/?o=7474649359993770.

To set up monitoring for your deployed agent, see:
https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring
